# Day 1–2 — Solar Energy Simulation & Green AI Scheduler

This notebook introduces the foundational energy model of the project.

The objective is to simulate solar energy availability over a full day and
to demonstrate how AI workloads can be explicitly constrained by energy
availability.

Rather than optimizing performance, this experiment focuses on:
- explainability,
- frugal computation,
- and energy-aware decision boundaries.

This energy-first logic will later govern all learning and decision modules
in the solar greenhouse system.


In [ ]:
# Core scientific libraries are sufficient for this experiment.
# No external dependencies are introduced in order to keep the simulation
# lightweight, reproducible, and compatible with low-resource environments.
#
# This design choice reflects the frugal AI philosophy of the project.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# A fixed random seed is used to ensure reproducibility of stochastic effects
# (e.g. simulated cloud variability).
np.random.seed(42)


In [ ]:
# Time is discretized over a 24-hour period with a 5-minute resolution.
#
# This resolution approximates realistic sensor sampling rates in
# small-scale greenhouses while keeping memory usage and computation low.
#
# Using a fixed temporal grid also simplifies later logging and analysis.
start = datetime(2025, 1, 1, 0, 0, 0)
minutes = np.arange(0, 24 * 60, 5)
times = [start + timedelta(minutes=int(m)) for m in minutes]

# Hours are represented as floating-point values (e.g. 13.5 = 13:30),
# which simplifies mathematical modeling of daily cycles.
t_hours = np.array([t.hour + t.minute / 60 for t in times])


In [ ]:
# Solar energy is modeled as a half-sine wave between sunrise and sunset.
#
# This simplified physical approximation captures the essential daily
# energy profile without relying on external weather or astronomical datasets.
#
# The goal is not physical accuracy, but explainability and controllability.
sunrise = 7.0
sunset = 19.0
daylen = sunset - sunrise

E = np.zeros_like(t_hours, dtype=float)
mask_day = (t_hours >= sunrise) & (t_hours <= sunset)

# Phase evolves from 0 to π during daylight hours
phase = (t_hours[mask_day] - sunrise) / daylen * np.pi
E[mask_day] = np.sin(phase)


In [ ]:
# To simulate real-world variability, stochastic noise ("clouds") is added.
#
# This noise introduces uncertainty in energy availability while remaining
# dataset-free and computationally inexpensive.
cloud_strength = 0.15
cloud_noise = np.random.normal(loc=0.0, scale=cloud_strength, size=E.shape)

# Noise is smoothed using a simple moving average to avoid unrealistic
# high-frequency fluctuations.
k = 5
kernel = np.ones(k) / k
cloud_noise_smoothed = np.convolve(cloud_noise, kernel, mode="same")

# Final solar energy signal, clipped to the [0, 1] interval
E_raw = E.copy()
E = np.clip(E + cloud_noise_smoothed, 0.0, 1.0)


In [ ]:
# An explicit energy threshold defines when AI workloads are allowed to run.
#
# This scheduler enforces an energy-first constraint:
# AI computation is permitted only when sufficient solar energy is available.
#
# This mimics the behavior of a solar-powered edge device and prevents
# uncontrolled computation during low-energy periods.
THRESHOLD = 0.5

def can_run_ai(energy, threshold=THRESHOLD):
    """
    Returns True if AI computation is allowed under the current
    energy constraint.
    """
    return energy >= threshold

can_run = np.array([can_run_ai(e) for e in E])

# Duty cycle represents the fraction of the day during which AI is active.
# It serves as a simple indicator of system autonomy under energy constraints.
duty_cycle = can_run.mean() * 100
duty_cycle


In [ ]:
# Visualization makes the energy constraint explicit and interpretable.
#
# The shaded regions correspond to time intervals where AI is allowed to run,
# clearly illustrating the relationship between solar energy and computation.
plt.figure(figsize=(12, 4))

plt.plot(t_hours, E, label="Solar energy E(t)")
plt.axhline(THRESHOLD, linestyle="--", label=f"Energy threshold = {THRESHOLD}")

for i in range(len(t_hours) - 1):
    if can_run[i]:
        plt.axvspan(t_hours[i], t_hours[i + 1], alpha=0.08)

plt.title(f"Simulated solar energy over one day • AI duty cycle ≈ {duty_cycle:.1f}%")
plt.xlabel("Hour of day")
plt.ylabel("Normalized solar energy")
plt.xlim(0, 24)
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# All energy values and AI activation decisions are logged to a CSV file.
#
# This log enables post-hoc analysis, explainability, and reproducibility,
# and will later be reused by decision agents and dashboards.
df = pd.DataFrame({
    "time": times,
    "hour_float": t_hours,
    "E_solar": E,
    "E_solar_ideal": E_raw,
    "can_run_ai": can_run
})

df["note"] = np.where(
    df["can_run_ai"],
    "AI active (sufficient energy)",
    "AI inactive (low energy)"
)

df.to_csv("solar_day_log.csv", index=False)
df.head()


Experiments — Why they exist


# Day length variation
# This experiment explores how changes in day length
# directly affect AI availability and autonomy.
#
# Longer daylight hours increase the AI duty cycle,
# illustrating the tight coupling between natural conditions
# and computational capacity.


# Cloudy day
# Increasing cloud density simulates adverse environmental conditions.
#
# This highlights how stochastic energy loss impacts AI availability
# and reinforces the need for adaptive, energy-aware strategies.


# Threshold comparison
# Comparing multiple energy thresholds illustrates the trade-off
# between conservative and aggressive AI behavior.
#
# Lower thresholds increase autonomy but risk energy instability,
# while higher thresholds favor safety at the cost of reduced activity.
